# Imputazione con MICE per i test ML

In [3]:
import re
import csv
from pathlib import Path

import warnings
import numpy as np
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# Durante l'addestramento dell'imputatore possono comparire warning di convergenza
# che non bloccano il flusso; li nascondo per mantenere l'output leggibile.
warnings.filterwarnings('ignore', category=ConvergenceWarning)



def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    # Risalgo la gerarchia delle cartelle finché non trovo sia il train set sia i dati corrotti.
    for candidate in [current, *current.parents]:
        train_file = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_imputation_train.csv'
        source_dir = candidate / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
        if train_file.exists() and source_dir.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto con i dati ML.')


project_root = find_project_root()
# Il modello viene stimato sul train set pulito; i test corrotti vengono solo imputati.
train_path = project_root / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_imputation_train.csv'
source_dir = project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML' / 'MICE'
file_pattern = 'heloc_ML_imputation_test_corrupted_*.csv'



def fit_mice_imputer(train_path: Path) -> tuple[list[str], list[str], IterativeImputer]:
    with train_path.open(newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        rows = list(reader)
        fieldnames = reader.fieldnames

    if not fieldnames:
        raise ValueError(f'File senza intestazione: {train_path}')

    # Separo le feature dalla variabile target, che non deve entrare nell'imputazione.
    feature_columns = [column for column in fieldnames if column != 'RiskPerformance']
    matrix = []

    # Converto i mancanti in NaN, perché l'imputatore lavora su valori numerici mancanti espliciti.
    for row in rows:
        numeric_row = []
        for column in feature_columns:
            value = row[column]
            numeric_row.append(np.nan if value == '' else float(value))
        matrix.append(numeric_row)

    # MICE stima ogni variabile in modo iterativo usando regressioni multivariate.
    imputer = IterativeImputer(
        estimator=BayesianRidge(),
        initial_strategy='median',
        max_iter=20,
        random_state=42,
        sample_posterior=False,
    )
    imputer.fit(matrix)

    return fieldnames, feature_columns, imputer



def impute_with_mice(
    input_path: Path,
    output_path: Path,
    fieldnames: list[str],
    feature_columns: list[str],
    imputer: IterativeImputer,
) -> dict:
    with input_path.open(newline='') as csv_file:
        reader = csv.DictReader(csv_file)
        rows = list(reader)
        input_fieldnames = reader.fieldnames

    if not input_fieldnames:
        raise ValueError(f'File senza intestazione: {input_path}')
    # Il test deve avere le stesse colonne del train set, altrimenti il modello non è applicabile.
    if input_fieldnames != fieldnames:
        raise ValueError(f'Intestazione non coerente tra training e test: {input_path}')

    matrix = []
    missing_before = 0

    # Trasformo i missing del test in NaN per poterli passare al modello già addestrato.
    for row in rows:
        numeric_row = []
        for column in feature_columns:
            value = row[column]
            if value == '':
                numeric_row.append(np.nan)
                missing_before += 1
            else:
                numeric_row.append(float(value))
        matrix.append(numeric_row)

    # Applico il modello sul test: non rifitto l'imputer, uso solo i parametri stimati sul train set.
    imputed_matrix = imputer.transform(matrix)

    # Rimpiazzo i missing con i valori stimati dal modello.
    for row_index, row in enumerate(rows):
        for column_index, column in enumerate(feature_columns):
            row[column] = str(imputed_matrix[row_index, column_index])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    return {
        'input_file': input_path.name,
        'output_file': output_path.name,
        'missing_before': missing_before,
        'missing_after': 0,
        'rows': len(rows),
        'iterations': imputer.n_iter_,
    }


# Stimo l'imputatore una sola volta sul train set e lo riutilizzo per tutti i test corrotti.
fieldnames, feature_columns, imputer = fit_mice_imputer(train_path)
results = []


def build_output_name(input_name: str) -> str:
    # Estrae dataset, strategia e percentuale dal nome del file corrotto e costruisce
    # il nome di output nel formato: [dataset]_discrirminative_train_[strategia]_[pct].csv
    match = re.match(r'^(.+?)_imputation_test_corrupted_([A-Z]+)_(\d+)\.csv$', input_name)
    if not match:
        raise ValueError(f'Nome file non riconosciuto: {input_name}')
    dataset, strategia, pct = match.group(1), match.group(2), match.group(3)
    return f'{dataset}_discrirminative_train_{strategia}_{pct}.csv'


for input_path in sorted(source_dir.glob(file_pattern)):
    output_name = build_output_name(input_path.name)
    output_path = output_dir / output_name
    results.append(impute_with_mice(input_path, output_path, fieldnames, feature_columns, imputer))

results

[{'input_file': 'heloc_ML_imputation_test_corrupted_MAR_10.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_10.csv',
  'missing_before': 13694,
  'missing_after': 0,
  'rows': 2958,
  'iterations': 7},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_25.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_25.csv',
  'missing_before': 29014,
  'missing_after': 0,
  'rows': 2958,
  'iterations': 7},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_40.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_40.csv',
  'missing_before': 43926,
  'missing_after': 0,
  'rows': 2958,
  'iterations': 7},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_10.csv',
  'output_file': 'heloc_ML_discrirminative_train_MCAR_10.csv',
  'missing_before': 13677,
  'missing_after': 0,
  'rows': 2958,
  'iterations': 7},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_25.csv',
  'output_file': 'heloc_ML_discrirminative_train_MCAR_25.csv',
  'missing_before': 2

In [4]:
# Genera un report aggregato dell'imputazione e lo salva in CSV
from pathlib import Path
import csv

# Se le variabili del notebook esistono, riusale; altrimenti ricostruiscile
try:
    _project_root = project_root
    _source_dir = source_dir
    _output_dir = output_dir
except NameError:
    def find_project_root(start: Path | None = None) -> Path:
        current = (start or Path.cwd()).resolve()
        for candidate in [current, *current.parents]:
            train_file = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_imputation_train.csv'
            source_dir_candidate = candidate / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
            if train_file.exists() and source_dir_candidate.exists():
                return candidate
        raise FileNotFoundError('Impossibile trovare la radice del progetto con i dati ML.')
    _project_root = find_project_root()
    _source_dir = _project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_ML'
    _output_dir = _project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML' / 'MICE'

report_path = _output_dir / 'imputation_report_ML_mice.csv'
rows_out = []

# Se 'results' è disponibile (variabile creata dalla cella di imputazione), usala direttamente
if 'results' in globals() and isinstance(results, list) and results:
    for r in results:
        rows_out.append({
            'input_file': r.get('input_file'),
            'output_file': r.get('output_file'),
            'rows': r.get('rows'),
            'missing_before': r.get('missing_before'),
            'missing_after': r.get('missing_after')
        })
else:
    # Ricostruisci il report leggendo i file imputati e confrontandoli con gli originali
    import re as _re
    for imputed in sorted(_output_dir.glob('heloc_ML_discrirminative_train_*.csv')):
        _m = _re.match(r'^(.+?)_discrirminative_train_([A-Z]+)_(\d+)\.csv$', imputed.name)
        corrupted_name = f'{_m.group(1)}_imputation_test_corrupted_{_m.group(2)}_{_m.group(3)}.csv' if _m else ''
        corrupted = _source_dir / corrupted_name
        rows_count = 0
        missing_after = 0
        missing_before = 0
        if imputed.exists():
            with imputed.open(newline='') as f:
                reader = csv.DictReader(f)
                imputed_rows = list(reader)
                rows_count = len(imputed_rows)
                # count empty strings after imputation (should be 0)
                for row in imputed_rows:
                    missing_after += sum(1 for v in row.values() if v == '')
        if corrupted.exists():
            with corrupted.open(newline='') as f:
                reader = csv.DictReader(f)
                corrupted_rows = list(reader)
                for row in corrupted_rows:
                    missing_before += sum(1 for v in row.values() if v == '')
        rows_out.append({
            'input_file': corrupted_name if corrupted.exists() else '',
            'output_file': imputed.name,
            'rows': rows_count,
            'missing_before': missing_before,
            'missing_after': missing_after
        })

# Scrivi il report CSV
_report_fields = ['input_file', 'output_file', 'rows', 'missing_before', 'missing_after']
_output_dir.mkdir(parents=True, exist_ok=True)
with report_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=_report_fields)
    writer.writeheader()
    writer.writerows(rows_out)

# Stampa un breve sommario
total_files = len(rows_out)
total_missing_before = sum(r['missing_before'] for r in rows_out)
print(f"Report salvato: {report_path}")
print(f"File processati: {total_files}")
print(f"Valori mancanti prima dell'imputazione (totale): {total_missing_before}")

rows_out  # ritorna il dettaglio come output dell'ultima expression

Report salvato: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Imputated_ML/MICE/imputation_report_ML_mice.csv
File processati: 9
Valori mancanti prima dell'imputazione (totale): 260571


[{'input_file': 'heloc_ML_imputation_test_corrupted_MAR_10.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_10.csv',
  'rows': 2958,
  'missing_before': 13694,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_25.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_25.csv',
  'rows': 2958,
  'missing_before': 29014,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MAR_40.csv',
  'output_file': 'heloc_ML_discrirminative_train_MAR_40.csv',
  'rows': 2958,
  'missing_before': 43926,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_10.csv',
  'output_file': 'heloc_ML_discrirminative_train_MCAR_10.csv',
  'rows': 2958,
  'missing_before': 13677,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imputation_test_corrupted_MCAR_25.csv',
  'output_file': 'heloc_ML_discrirminative_train_MCAR_25.csv',
  'rows': 2958,
  'missing_before': 29034,
  'missing_after': 0},
 {'input_file': 'heloc_ML_imput